# Chapter 10, Application 3: Linear systems beyond HHL

Companion notebook to *Quantum Algorithms from First Principles*, Chapter 10,
"Building the QSVT linear solver".

Steps: (1) bound $\kappa$ to fix the approximation domain
$x\in[-1,-1/\kappa]\cup[1/\kappa,1]$; (2) fit an odd polynomial to $1/x$ there and rescale
so $|P|\le1$ on all of $[-1,1]$, the forbidden gap included; (3) phase factors offline;
(4) wrap in $O(\kappa)$ rounds of amplitude amplification. Tests: the polynomial against
$1/x$ classically, $|P|\le1$ everywhere, the projected block against $A^{-1}$ rescaled, and
the final state against $A^{-1}|b\rangle$ up to global phase.

The block encoding is the book's one-ancilla Hermitian form
$U=\begin{pmatrix}A&\sqrt{I-A^2}\\\sqrt{I-A^2}&-A\end{pmatrix}$, taken as a black-box
unitary gate. Phase finding and the QSVT circuit are those of the qubitization notebook.
Convention: book wire $j$ is Qiskit qubit `bq(j)`.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import sqrtm
from scipy.optimize import minimize
from numpy.polynomial import chebyshev as Ch
from qiskit import QuantumCircuit
from qiskit.circuit.library import UnitaryGate
from qiskit.quantum_info import Operator, Statevector

bq = lambda j, tot: tot - 1 - j

## Step 1: the system and its condition number

In [2]:
rng = np.random.default_rng(5)
Q, _ = np.linalg.qr(rng.normal(size=(4, 4)))
eigs = np.array([-1.0, -0.4, 0.3, 0.8])
A = Q @ np.diag(eigs) @ Q.T
b = rng.normal(size=4); b /= np.linalg.norm(b)
lam = np.linalg.eigvalsh(A)
kappa = np.max(np.abs(lam)) / np.min(np.abs(lam))
# Gershgorin bound on |lambda|_max, the cheap route the book mentions
gersh = np.max(np.abs(np.diag(A)) + np.sum(np.abs(A - np.diag(np.diag(A))), axis=1))
print("eigenvalues:", np.round(lam, 4), f" kappa = {kappa:.3f}, Gershgorin bound on |lambda| = {gersh:.3f}")
kappa_used = 3.5                                  # a slight overestimate of kappa: safe side
x_min = 1 / kappa_used

eigenvalues: [-1.  -0.4  0.3  0.8]  kappa = 3.333, Gershgorin bound on |lambda| = 1.268


## Step 2: an odd polynomial for $1/(\kappa x)$ on the domain, rescaled so $|P|\le1$ everywhere

In [3]:
d = 21                                            # odd degree
xs_dom = np.concatenate([-np.linspace(x_min, 1, 300), np.linspace(x_min, 1, 300)])
target_dom = 1 / (kappa_used * xs_dom)
# least-squares fit in the odd Chebyshev basis
basis = np.array([Ch.chebval(xs_dom, [0] * k + [1]) for k in range(1, d + 1, 2)]).T
coef_odd, *_ = np.linalg.lstsq(basis, target_dom, rcond=None)
cheb = np.zeros(d + 1); cheb[1::2] = coef_odd
P = lambda x: Ch.chebval(x, cheb)

xs_all = np.linspace(-1, 1, 2001)
rel_err = np.max(np.abs(P(xs_dom) - target_dom) / np.abs(target_dom))
max_all = np.max(np.abs(P(xs_all)))
print(f"degree {d}: relative error against 1/(kappa x) on the domain {rel_err:.2e}; max |P| on [-1,1] = {max_all:.3f}")
scale = 1.0
if max_all > 1:
    scale = 0.999 / max_all
    cheb *= scale
    print(f"rescaled by {scale:.4f} so that |P| <= 1 including the gap")
assert np.max(np.abs(P(xs_all))) <= 1
plt.plot(xs_all, P(xs_all), label=f"P, degree {d}")
plt.plot(xs_dom, scale / (kappa_used * xs_dom), ".", ms=2, label=r"$1/(\kappa x)$ on the domain")
plt.axvspan(-x_min, x_min, color="0.9", label="forbidden gap"); plt.ylim(-1.1, 1.1); plt.legend(); plt.show()

degree 21: relative error against 1/(kappa x) on the domain 8.42e-03; max |P| on [-1,1] = 1.188
rescaled by 0.8407 so that |P| <= 1 including the gap


## Step 3: phase factors, offline

In [4]:
def qsvt_scalar_vec(phases, xs):
    xs = np.asarray(xs, dtype=float); s = np.sqrt(1 - xs**2)
    Ux = np.stack([np.stack([xs, s], -1), np.stack([s, -xs], -1)], -2)
    D = lambda p: np.array([[np.exp(1j * p), 0], [0, np.exp(-1j * p)]])
    M = np.broadcast_to(D(phases[0]), Ux.shape).copy()
    for p in phases[1:]:
        M = M @ Ux @ D(p)
    return M[:, 0, 0]


def find_phases(target, d, n_restarts=6, seed=0):
    xs = np.cos(np.pi * (np.arange(2 * d + 2) + 0.5) / (2 * d + 2))
    ts = target(xs)
    half = (d + 2) // 2
    expand = (lambda h: np.concatenate([h, h[::-1]])) if d % 2 == 1 else (lambda h: np.concatenate([h, h[-2::-1]]))
    loss = lambda h: np.sum((np.real(qsvt_scalar_vec(expand(h), xs)) - ts) ** 2)
    rng = np.random.default_rng(seed); best = None
    for h0 in [np.full(half, np.pi / 2)] + [rng.uniform(0, np.pi, half) for _ in range(n_restarts)]:
        r = minimize(loss, h0, method="L-BFGS-B", options=dict(ftol=1e-16, gtol=1e-12, maxiter=10000))
        if best is None or r.fun < best.fun: best = r
        if best.fun < 1e-20: break
    return expand(best.x), best.fun


phases, resid = find_phases(P, d)
print(f"fit residual {resid:.1e}; max |Re P_phi - P| on [-1,1] = {np.max(np.abs(np.real(qsvt_scalar_vec(phases, xs_all)) - P(xs_all))):.1e}")

fit residual 9.8e-13; max |Re P_phi - P| on [-1,1] = 3.7e-07


## The circuit: block encoding, QSVT with the real-part trick, and $|b\rangle$

In [5]:
n_sys = 2; N = 4
U_mat = np.block([[A, sqrtm(np.eye(N) - A @ A)], [sqrtm(np.eye(N) - A @ A), -A]]).real
assert np.allclose(U_mat @ U_mat.T, np.eye(2 * N)) and np.allclose(U_mat[:N, :N], A)
tot = 1 + n_sys                                   # block-encoding ancilla = book wire 0
idx = [bq(0, tot)]
sysq = [bq(1 + j, tot) for j in range(n_sys)]
U_gate = UnitaryGate(U_mat, label="U_BE")         # matrix is in book order; Qiskit wants LSB first


def append_U(qc, qubits_book_order):
    qc.append(U_gate, qubits_book_order[::-1])


def Pi_phi(qc, anc, phi):
    qc.x(anc); qc.p(2 * phi, anc); qc.x(anc); qc.global_phase += -phi


def qsvt_circuit(phases):
    qc = QuantumCircuit(tot)
    for phi in reversed(phases[1:]):
        Pi_phi(qc, idx[0], phi); append_U(qc, idx + sysq)
    Pi_phi(qc, idx[0], phases[0])
    return qc


def solver_block(phases):
    """2 + n_sys qubits: real-part ancilla (wire 0), BE ancilla (wire 1), system."""
    t2 = 1 + tot
    c = bq(0, t2); rest = list(range(t2 - 1))
    qc = QuantumCircuit(t2)
    Up = qsvt_circuit(phases).to_gate(label="U_phi"); Um = qsvt_circuit(-np.array(phases)).to_gate(label="U_-phi")
    qc.h(c); qc.x(c); qc.append(Up.control(1), [c] + rest); qc.x(c); qc.append(Um.control(1), [c] + rest); qc.h(c)
    return qc


blk = Operator(solver_block(phases)).data[:N, :N]
A_inv_scaled = scale * np.linalg.inv(A) / kappa_used
w_, V_ = np.linalg.eigh(A)
P_of_A = V_ @ np.diag(P(w_)) @ V_.T
print(f"Test 3: ||block - P(A)|| = {np.linalg.norm(blk - P_of_A, 2):.1e}  (circuit and phases)")
print(f"        ||block - A^-1 scale/kappa|| = {np.linalg.norm(blk - A_inv_scaled, 2):.1e}  (that is the polynomial's own {rel_err:.1e} relative error)")
assert np.linalg.norm(blk - P_of_A, 2) < 1e-5
assert np.linalg.norm(blk - A_inv_scaled, 2) < 2 * rel_err * scale

Test 3: ||block - P(A)|| = 3.5e-07  (circuit and phases)
        ||block - A^-1 scale/kappa|| = 2.0e-03  (that is the polynomial's own 8.4e-03 relative error)


## Test 4: the output state against $A^{-1}|b\rangle$, and the success probability

In [6]:
t3 = 2 + n_sys
sys3 = [bq(2 + j, n_sys + 2) for j in range(n_sys)]


def full_circuit(phases):
    qc = QuantumCircuit(t3)
    qc.prepare_state(b, sys3[::-1])
    qc.compose(solver_block(phases), inplace=True)
    return qc


sv = Statevector(full_circuit(phases))
amps = sv.data.reshape([2] * t3)
out = amps[0, 0].reshape(-1)                       # both ancillas = 0
p_success = np.linalg.norm(out) ** 2
x_circ = out / np.linalg.norm(out)
x_exact = np.linalg.solve(A, b); x_exact /= np.linalg.norm(x_exact)
fid = abs(np.vdot(x_exact, x_circ)) ** 2
print(f"fidelity with A^-1|b>: {fid:.8f}; success probability {p_success:.4f} (||A^-1 b||^2 scale^2/kappa^2 = {np.linalg.norm(np.linalg.inv(A)@b)**2*scale**2/kappa_used**2:.4f})")
assert fid > 1 - 1e-4

fidelity with A^-1|b>: 0.99999687; success probability 0.4109 (||A^-1 b||^2 scale^2/kappa^2 = 0.4110)


## Step 4: amplitude amplification on the two ancillas

$\mathcal{A}$ is the whole circuit above, the good subspace is "both ancillas read 0",
and each round runs $\mathcal{A}$ and $\mathcal{A}^\dagger$ once. With $\theta=\arcsin\sqrt p$,
$r=\lfloor\pi/(4\theta)\rfloor$ rounds bring the success probability as close to 1 as the
discrete rotation allows (overshooting past $r$ lowers it again, as in Chapter 6). This
small system is well conditioned, so $p$ starts around $0.4$ and one round suffices; in
general $p=O(1/\kappa^2)$ and $O(\kappa)$ rounds replace $O(\kappa^2)$ repetitions.

In [7]:
A_circ = full_circuit(phases).to_gate(label="A")
A_dag = A_circ.inverse()
anc2 = [bq(0, t3), bq(1, t3)]


def S_good(qc):            # phase -1 on ancillas = 00
    qc.x(anc2); qc.cz(anc2[0], anc2[1]); qc.x(anc2)


def S_zero(qc):            # phase -1 on the all-zero state of all t3 qubits
    qc.x(range(t3)); qc.h(t3 - 1); qc.mcx(list(range(t3 - 1)), t3 - 1); qc.h(t3 - 1); qc.x(range(t3))


def amplified(r):
    qc = QuantumCircuit(t3)
    qc.append(A_circ, range(t3))
    for _ in range(r):
        S_good(qc)
        qc.append(A_dag, range(t3)); S_zero(qc); qc.append(A_circ, range(t3))
    return qc


theta = np.arcsin(np.sqrt(p_success))
r_opt = int(np.floor(np.pi / (4 * theta)))
for r in range(r_opt + 2):
    sv = Statevector(amplified(r)); amps = sv.data.reshape([2] * t3); out = amps[0, 0].reshape(-1)
    p = np.linalg.norm(out) ** 2; f = abs(np.vdot(x_exact, out / np.linalg.norm(out))) ** 2
    print(f"rounds {r}: success probability {p:.4f} (predicted sin^2((2r+1)theta) = {np.sin((2*r+1)*theta)**2:.4f}), fidelity {f:.6f}")
print(f"optimal rounds r = {r_opt}; each round = {2 * d} uses of the block encoding on top of the {d} in A itself")

rounds 0: success probability 0.4109 (predicted sin^2((2r+1)theta) = 0.4109), fidelity 0.999997


rounds 1: success probability 0.7560 (predicted sin^2((2r+1)theta) = 0.7560), fidelity 0.999997


rounds 2: success probability 0.1096 (predicted sin^2((2r+1)theta) = 0.1096), fidelity 0.999997
optimal rounds r = 1; each round = 42 uses of the block encoding on top of the 21 in A itself
